# DATALUS OAA Compliance: Auditoria de Privacidade Empírica e Utilidade

Este notebook demonstra o **Orquestrador de Auditoria Autônomo (OAA)**.
Avaliamos os dados sintéticos gerados em relação a benchmarks rigorosos de privacidade e utilidade para comprovar sua segurança para lançamento público
sob regulamentações de proteção de dados (ex: LGPD/GDPR).

A auditoria inclui simulações de **Ataque de Inferência de Pertencimento (MIA)** e verificações de **Distância para o Registro mais Próximo (DCR)** para memorização.

## 1. Configuração do Ambiente e Verificação de Dados
Detectando o ambiente e verificando se os conjuntos de dados reais e sintéticos estão prontos para a auditoria.
Se os dados estiverem ausentes, acionamos um processo de geração de fallback.

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import numpy as np

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_audit'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_audit'
    return './datalus_audit'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

INSTALLED = Path(WORKING_DIR) / '.installed'

if not INSTALLED.exists():
    if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        !git clone --depth 1 https://github.com/emanuellcs/datalus.git "{WORKING_DIR}/source" 2>/dev/null || true
        %cd "{WORKING_DIR}/source"
        !pip install --quiet -e '.[dev]' pysus polars matplotlib seaborn scikit-learn nest_asyncio 2>&1 | tail -5
        sys.path.insert(0, f'{WORKING_DIR}/source')
        INSTALLED.touch()
        print()
        print('=' * 60)
        print('  INSTALAÇÃO CONCLUÍDA')
        print('  Clique em "RESTART RUNTIME" no aviso acima')
        print('  ou vá em Ambiente de execução > Reiniciar e executar tudo')
        print('=' * 60)
        os._exit(0)
    else:
        INSTALLED.touch()
        print('Ambiente local. Dependências consideradas instaladas.')
else:
    src = Path(WORKING_DIR) / 'source'
    if src.is_dir():
        sys.path.insert(0, str(src))
    print('Dependências já instaladas.')

real_path = Path(f'{WORKING_DIR}/processed.parquet')
synth_path = Path(f'{WORKING_DIR}/synthetic.parquet')
schema_path = Path(f'{WORKING_DIR}/schema_config.json')

if not (real_path.exists() and synth_path.exists()):
    print('Data not found. Generating fallback dummy datasets for audit demo...')
    from pysus import sih, sim, sinan
    from datetime import datetime
    import urllib.request
    import asyncio

    STATES = ['SP', 'RJ', 'MG', 'PR', 'RS', 'BA', 'DF']

    def fmt_tier(label, state, year, month, count):
        return f'Tier {label}: {state}/{year}/{month} ({count} records)'

    def try_pysus():
        now = datetime.now()
        sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
        for state in STATES:
            for y in range(sy, sy - 2, -1):
                for m in range(sm if y == sy else 12, 0, -1):
                    try:
                        df = sih(state=state, year=y, month=[m], group='RD', as_dataframe=True)
                        if df is not None and len(df) > 0 and 'MORTE' in df.columns:
                            print(fmt_tier('1: SIH-RD', state, y, m, len(df)))
                            return pl.from_pandas(df)
                    except Exception:
                        continue
            for y in range(sy, sy - 2, -1):
                try:
                    df = sim(state=state, year=y, as_dataframe=True)
                    if df is not None and len(df) > 0:
                        print(fmt_tier('2: SIM', state, y, 'all', len(df)))
                        df['MORTE'] = 1
                        return pl.from_pandas(df)
                except Exception:
                    continue
        for y in range(sy, sy - 2, -1):
            try:
                df = sinan(disease='deng', year=y, as_dataframe=True)
                if df is not None and len(df) > 0:
                    print(fmt_tier('3: SINAN-deng', 'BR', y, 'all', len(df)))
                    if 'MORTE' not in df.columns:
                        df['MORTE'] = 0
                    return pl.from_pandas(df)
            except Exception:
                continue
        return None

    def try_direct_ftp():
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except ImportError:
            return None
        from pysus.api.extensions import DBC
        now = datetime.now()
        sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
        base = 'ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/199301_/Dados/'
        for state in ['SP', 'RJ', 'MG']:
            for y in range(sy, sy - 2, -1):
                for m in range(sm if y == sy else 12, 0, -1):
                    fname = f'RD{state}{y % 100:02d}{m:02d}.dbc'
                    try:
                        with urllib.request.urlopen(base + fname, timeout=30) as r:
                            dbc_path = Path('/tmp') / fname
                            dbc_path.write_bytes(r.read())
                        df = asyncio.run(DBC(path=dbc_path).load())
                        if df is not None and len(df) > 0:
                            print(fmt_tier('4: FTP direct', state, y, m, len(df)))
                            return pl.from_pandas(df)
                    except Exception:
                        continue
        return None

    def generate_sample():
        print('Tier 5: Generated demo sample (real data unavailable)')
        print('NOTE: This is not real data — results are for pipeline demo only.')
        np.random.seed(42)
        n = 1000
        return pl.DataFrame({
            'MORTE': np.random.choice([0, 1], n, p=[0.92, 0.08]).astype(np.int64),
            'IDADE': np.random.randint(0, 100, n).astype(np.float64),
            'SEXO': np.random.choice(['0', '1'], n),
            'DIAS_PERMANENCIA': np.clip(np.random.poisson(5, n), 0, 100).astype(np.float64),
            'VALOR_TOTAL': np.round(np.random.lognormal(7, 1.5, n), 2),
            'PROCEDIMENTO_PRINCIPAL': np.random.choice(['AB', 'CD', 'EF', 'GH', 'IJ'], n),
            'MUNICIPIO_MOV': np.random.choice(['3550308', '3304557', '3106200'], n),
        })

    df = try_pysus()
    if df is None:
        df = try_direct_ftp()
    if df is None:
        df = generate_sample()

    df = df.with_columns(
        pl.col('MORTE').cast(pl.Int64).fill_null(0)
    )

    cols_to_keep = [
        c for c in df.columns
        if not c.startswith(('N_AIH', 'IDENT', 'CNS', 'CPF', 'CNPJ'))
    ]
    df = df.select(
        ['MORTE'] + [c for c in cols_to_keep if c != 'MORTE']
    ).head(2000)
    df.write_parquet(f'{WORKING_DIR}/raw_sample.parquet')
    
    !datalus ingest {WORKING_DIR}/raw_sample.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE
    !datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 1 --batch-size 256
    !datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 2000
else:
    print('Verified: Audit datasets are ready.')


## 2. Executando o Orquestrador de Auditoria Autônomo (OAA)
Executamos o comando `datalus audit`. Para esta demonstração, usamos um subconjunto limitado de linhas para garantir a execução rápida.
A flag `--mia-mode release` aciona a simulação completa de ataque com modelos de sombra (shadow models).

In [ ]:
!datalus audit {WORKING_DIR}/processed.parquet {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/schema_config.json {WORKING_DIR}/audit_report.json --target-column MORTE --mia-mode release

## 3. Analisando e Interpretando os Resultados da Auditoria
Carregando o relatório JSON e extraindo o veredito de Passa/Falha de alto nível com base nos limiares de privacidade e utilidade.

In [ ]:
import json
with open(f'{WORKING_DIR}/audit_report.json', 'r') as f:
    report = json.load(f)

verdict = "PASSED" if report.get('overall_verdict', False) else "FAILED"
print(f"--- AUDIT VERDICT: {verdict} ---")
print(f"Privacy Score: {report.get('privacy', {}).get('mia_roc_auc', 'N/A')}")
print(f"Utility MLE Ratio: {report.get('utility', {}).get('mle_ratio', 'N/A')}")


## 4. Validação de Privacidade: Distância para o Registro mais Próximo (DCR)
O DCR mede se o modelo memorizou registros reais específicos. Verificamos a distância do 5º percentil em relação ao limiar de alerta.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'privacy' in report and 'dcr_stats' in report['privacy']:
    dcr = report['privacy']['dcr_stats']
    p5_dcr = dcr.get('p5', 0)
    threshold = 0.05 # Exemplo de limiar
    
    plt.figure(figsize=(10, 5))
    colors = ['green' if p5_dcr > threshold else 'red']
    plt.bar(['5th Percentile DCR'], [p5_dcr], color=colors)
    plt.axhline(y=threshold, color='black', linestyle='--', label='Security Threshold')
    plt.title('Privacy Check: Distance to Closest Record (Memorization Risk)')
    plt.ylabel('Normalized Distance')
    plt.legend()
    plt.show()


## 5. Validação de Privacidade: AUC de Ataque Shadow-MIA
Uma AUC próxima de 0.50 indica que o atacante não consegue distinguir entre membros e não membros do conjunto de treinamento.

In [ ]:
if 'privacy' in report and 'mia_roc_auc' in report['privacy']:
    auc = report['privacy']['mia_roc_auc']
    
    plt.figure(figsize=(10, 5))
    plt.bar(['MIA ROC-AUC'], [auc], color='orange')
    plt.axhline(y=0.5, color='blue', linestyle='--', label='Random Guess (Perfect Privacy)')
    plt.axhline(y=0.6, color='red', linestyle='--', label='Alert Threshold')
    plt.ylim(0.4, 1.0)
    plt.title('Privacy Check: Membership Inference Attack Resistance')
    plt.ylabel('ROC-AUC Score')
    plt.legend()
    plt.show()


## 6. Validação de Utilidade: Eficácia de Machine Learning TSTR vs TRTR
A razão MLE comprova que modelos treinados em dados sintéticos têm desempenho quase tão bom quanto aqueles treinados em dados reais.

In [ ]:
if 'utility' in report and 'mle_scores' in report['utility']:
    mle = report['utility']['mle_scores']
    labels = ['TSTR', 'TRTR']
    scores = [mle.get('tstr', 0), mle.get('trtr', 0)]
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=labels, y=scores, palette='viridis')
    plt.title(f"Utility Check: MLE Ratio ({report['utility'].get('mle_ratio', 0):.2f})")
    plt.ylabel('F1 / Accuracy Score')
    plt.show()
